### Step 1: Import python libraries and api key

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Set up Pushover
* Step 2a: set up account in your browser (https://pushover.net)
* Step 2b: set up the app on your iPhone/Adroid phone, long into the same account
* Step 2c: in the browser create an "Application/API Toke"
* Step 2d: copy your User Key and API Token into the .env file like the following but without the dot 
* PUSHOVER_USER=XXXXXXXXXXXXXXXXX
* PUSHOVER_TOKEN=YYYYYYYYYYYYYYYYYYY

* Save changes to the .env file
* Run a test by manually sending a notification from Pushover in your browser to your phone. 

In [2]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [3]:
# test Pushover
import requests

def send_notification(message: str):
    payload={"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
@send_notification("Hello to myself from Pushover.")

### Step 3: Describe Pushover as an LLM tool

In [4]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notification to the user's phone via Pushover. User this to alert the user of time-sensitive issues.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pusherover to the list of tools for the LLM

In [5]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 5: Call the tool from an LLM

In [6]:
def handle_tool_call(tool_calls):
    tool_call=tool_calls[0]  # assuming just one tool call
    args=json.loads(tool_call.function.arguments)
    
    # Actually send the notification for the tool call
    send_notification(args["message"])
    #print(f"Sent nofitification: {args['message']}")

    tool_call_result={
        "role": "tool",
        "content": f"Notification sent:, {args['message']}",
        "tool_call_id": tool_call.id
    }

    return tool_call_result

In [7]:
client=OpenAI()
messages=[
    {"role": "user", "content": "Please tell me the amazing progress I have made in the ai engineering class, and send a message via Pushover to my phone."
    }
    ]

response=client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=messages, 
    tools=tools
)

# Check if the model wants to call a tool
message = response.choices[0].message
print(f"Round 1: {message}")
print(f"Round 1 tool calls the model want to make: {message.tool_calls}")
print(f"Round 1 role of the message: {message.role}")


# check if model wants to call a tool
if message.tool_calls:
    #... handle tool call
    tool_result=handle_tool_call(message.tool_calls) # whole list of tool calls on purpose
    print(f"Tool Result: {tool_result}")
    #... add message to "context", i.e. messages
    messages.append(message)
    #... add info about tool call response to "context", i.e. messages
    messages.append(tool_result)
    #... invoke the LLM one more time to get updated response
    response=client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
        #tools=tools #will add this in the future
    )
    #... print(message.content) from the new llm response
    message=response.choices[0].message
    print(f"Round 2 tool calls the model want to make: {message.tool_calls}")
    print(f"Round 2 role of the message: {message.role}")
    print(f"Round 2 after tool call: {message.content}")

else: 
    print(f"No tool call. Reply: {message.content}")


Round 1: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_35otTI8tuwZoSuImavvcBfvN', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI engineering class! Your hard work and dedication are truly paying off. Keep it up!"}', name='send_notification'), type='function')])
Round 1 tool calls the model want to make: [ChatCompletionMessageFunctionToolCall(id='call_35otTI8tuwZoSuImavvcBfvN', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI engineering class! Your hard work and dedication are truly paying off. Keep it up!"}', name='send_notification'), type='function')]
Round 1 role of the message: assistant
Tool Result: {'role': 'tool', 'content': 'Notification sent:, Congratulations on your amazing progress in the AI engineering class! Your hard work and dedication are truly paying off.

In [30]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_stoWNOSho9TVUkVoGq1wG5dO', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!"}', name='send_notification'), type='function')])


In [32]:
if message.tool_calls:
    tool_call=message.tool_calls[0]
    import json
    args=json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Send notification: {args["message"]}")

else: 
    print(message.content)



Send notification: Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!
